In [0]:
%sql
USE CATALOG industry

# Clean the Raw Data

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW location_silver AS
WITH clean_data AS (
  SELECT
    country_id,
    CAST(SPLIT(country_id, "-")[1] AS BIGINT) AS id,
    CASE
      WHEN country_name = 'DE' THEN 'Germany'
      WHEN country_name in ('US', 'USA') THEN 'United States'
      ELSE TRIM(country_name)
    END AS country_name,
    etl_filename
  FROM
    bronze.location
)
SELECT
  id as country_id,
  country_name,
  country_id AS etl_business_key,
  SHA2(CONCAT(country_id, country_name), 256) AS etl_record_hash,
  etl_filename
FROM
  clean_data
WHERE
  country_name != ""
  AND country_name IS NOT NULL

# MERGE TO Silver Table

In [0]:
%sql
MERGE INTO
  silver.country as tgt
USING
  location_silver as src
ON
  tgt.etl_business_key = src.etl_business_key
  AND tgt.etl_record_hash <> src.etl_record_hash
WHEN MATCHED THEN UPDATE SET
  tgt.country_name = src.country_name,
  tgt.etl_record_hash = src.etl_record_hash,
  tgt.etl_update_timestamp = NOW()
WHEN NOT MATCHED THEN INSERT (
    tgt.country_id,
    tgt.country_name,
    tgt.etl_business_key,
    tgt.etl_record_hash,
    tgt.etl_filename
  )
  VALUES (
    src.country_id,
    src.country_name,
    src.etl_business_key,
    src.etl_record_hash,
    src.etl_filename
  )